# Monthly Aggregation

This recipe shows how to control **temporal frequency** and **summary statistics** when computing EVI zonal statistics. The quickstart uses evy's defaults (`freq="ME"`, `stats="mean"`); here we vary both parameters to build a richer time series.

For setup instructions, see the [quickstart](../quickstart.ipynb).

In [1]:
# papermill parameters
country = "RWA"
admin_level = 1
start_date = "2022-01-01"
end_date = "2024-12-31"
quick_mode = False


In [2]:
if quick_mode:
    start_date = "2023-01-01"
    end_date = "2023-12-31"


In [3]:
import evy

## Load boundaries

We use Rwanda's 5 provinces (ADM1) as our analysis regions.

In [4]:
gdf = evy.get_boundaries(country, admin_level=admin_level)
gdf[["shapeName", "shapeISO"]].head()

Found legacy boundaries cache at /Users/farhanreynaldo/Documents/world-bank/git-repo/evy/notebooks/recipes/.evy/boundaries. evy now caches to /Users/farhanreynaldo/Library/Caches/evy/boundaries; you can safely delete the legacy directory.


,shapeName,shapeISO
0,City of Kigali,RW-01
1,Southern Province,RW-05
2,Northern Province,RW-03
3,Eastern Province,RW-02
4,Western Province,RW-04


## Compute monthly statistics

The `freq` parameter accepts `"Original"` (the source composites), `"ME"` (monthly), `"QE"` (quarterly), and `"YE"` (yearly). Periods follow the calendar, and each row's `date` is the start of its period. evy also provides named constants for these: `evy.ORIGINAL`, `evy.MONTHLY`, `evy.QUARTERLY`, `evy.YEARLY`.

The `stats` parameter takes a string or list: `"mean"`, `"median"`, `"min"`, `"max"`, `"std"`, `"sum"`, `"count"`.

Let's request monthly mean and standard deviation over three years:

In [5]:
df = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date=start_date,
    end_date=end_date,
    freq=evy.MONTHLY,
    stats=["mean", "std"],
)
df.head(10)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


,date,shapeName,mean,std
0,2022-01-01,City of Kigali,0.315163,0.060495
1,2022-01-01,Southern Province,0.367447,0.048062
2,2022-01-01,Northern Province,0.381273,0.054762
3,2022-01-01,Eastern Province,0.359908,0.061724
4,2022-01-01,Western Province,0.384219,0.057469
5,2022-02-01,City of Kigali,0.323696,0.057492
6,2022-02-01,Southern Province,0.366557,0.046824
7,2022-02-01,Northern Province,0.378534,0.054335
8,2022-02-01,Eastern Province,0.341100,0.054171
9,2022-02-01,Western Province,0.382150,0.058738


## Visualize the time series

`plot_time_series` shows the EVI signal over time. With 3 years of monthly data, seasonal patterns become visible.

In [6]:
evy.plot_time_series(df, value_col="mean")

alt.LayerChart(...)

## Compare frequencies

To see the effect of temporal resolution, let's compute quarterly aggregation over the same period and plot both side by side.

In [7]:
df_quarterly = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date=start_date,
    end_date=end_date,
    freq=evy.QUARTERLY,
    stats=["mean"],
)
df_quarterly.head()

,date,shapeName,mean
0,2022-01-01,City of Kigali,0.329610
1,2022-01-01,Southern Province,0.372468
2,2022-01-01,Northern Province,0.383562
3,2022-01-01,Eastern Province,0.352709
4,2022-01-01,Western Province,0.383397


In [8]:
evy.plot_time_series(
    df_quarterly, value_col="mean", title="EVI Time Series (Quarterly)"
)

alt.LayerChart(...)

Monthly data preserves the intra-seasonal shape (green-up and senescence), while quarterly data smooths it into broad trends. For agricultural monitoring, monthly (`"ME"`) is usually the best trade-off between resolution and noise — see [Design Decisions](../../docs/design-decisions.md) for the rationale.